# EduHarness — Stage 3: Optional Colab Compatibility

The default workflow runs offline stage `3` and `python -m http.server` locally after all reviews pass. This notebook remains available only when Colab bundling/preview is useful.

Stage `3` does not require `OPENAI_API_KEY`. It packages `manifest.json`, media, interactive JSON, and the self-contained player.


In [ ]:
from __future__ import annotations
from pathlib import Path

from eduharness.colab_session import load_session
from eduharness.config import Config
from eduharness.pipeline import load_request, stage3

session = load_session()
request = load_request(Path(session["REQUEST"]))
cfg = Config.from_env(
    request,
    run_dir=Path(session["RUN_DIR"]) / request.request_id,
    require_api_key=False,
)
manifest = stage3(cfg)
print(manifest.model_dump_json(indent=2)[:1500])
print("Player:", cfg.bundle_dir / "index.html")


In [ ]:
# Optional preview via local HTTP + Colab proxy
from google.colab import output  # type: ignore
from pathlib import Path
import json

env = json.loads(Path("/content/eduharness_colab_env.json").read_text())
from eduharness.pipeline import load_request
from eduharness.config import Config

request = load_request(Path(env["REQUEST"]))
cfg = Config.from_env(
    request,
    run_dir=Path(env["RUN_DIR"]) / request.request_id,
    require_api_key=False,
)
bundle = cfg.bundle_dir

class Handler(SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory=str(bundle), **k)

server = ThreadingHTTPServer(("0.0.0.0", 8765), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("Serving", bundle)
display(output.serve_kernel_port_as_iframe(8765, height=720))
